# Modules and Functions

## Data

In [1]:
import pandas as pd

# read in the data:
city_df = pd.read_csv("data/geonames-all-cities-with-a-population-1000.csv", sep=";")

# ignore small "cities"
# city_df = city_df[city_df["Population"]>10000]

# There's a Baden in Austria, but we most likely don't want that one...
city_df = city_df.drop(city_df[(city_df["Name"] == "Baden") & (city_df["Country Code"] == "AT")].index)

# This city Aiud can also be called strassburg, but we don't want that:
city_df.loc[city_df["Name"] == "Aiud", "Alternate Names"] = city_df.loc[city_df["Name"] == "Aiud", "Alternate Names"].str.replace(r'Strassburg|Straßburg', '', regex=True)

# cities that Nägeli had corresponance with:
corres_cities_df = pd.read_csv("data/naegeli_corres_cities.csv", sep=";")
corres_cities = list(set(list(corres_cities_df['Ort'])))  # unique list

In [2]:
from scripts.utils import create_geo_df
data_df, cities_not_found, cities_multiple_matches = create_geo_df(corres_cities, city_df)
data_df.head()

  0%|          | 0/216 [00:00<?, ?it/s]

100%|██████████| 216/216 [00:06<00:00, 35.61it/s]


,city_orig_name,city,lon,lat
0,Meersburg,Meersburg,9.27113,47.69419
1,Orléans,Orléans,1.90389,47.90289
2,Prag,Prague,14.42076,50.08804
3,Wettingen,Wettingen,8.32663,47.46606
4,Lenzburg,Lenzburg,8.17503,47.38853


In [3]:
cities_multiple_matches

[['Frankfurt', ['Frankfurt am Main', 'Frankfurt (Oder)']],
 ['Strassburg', ['Strasbourg', 'Kuchurhan', 'Straßburg-Stadt']],
 ['Neuenburg',
  ['Neuenburg am Rhein',
   'Jaunpils',
   'Nowogródek Pomorski',
   'Neuchâtel',
   'Neuenbürg']],
 ['Kempten', ['Kempten (Allgäu)', 'Kempten (Allgäu)']]]

## Drawing the lines
If you draw lines, do it before the dots, otherwise the hover function won't work.

In [4]:
from scripts.utils import create_arrow_data
import plotly.graph_objects as go

# Startet immer von Zürich
zh_coor = (8.55, 47.36667)
lons, lats = create_arrow_data(zh_coor, data_df)

fig = go.Figure()

fig.add_trace(
    go.Scattergeo(
        lon = lons,
        lat = lats,
        mode = 'lines',
        hoverinfo = "skip",
        line = dict(width = 0.5,color = 'red'),
        opacity = 0.5,
    )
)

fig.show()

## Drawing Dots

In [5]:

# fig = go.Figure()  # Uncomment, if no Arrow data generated

# Add the cities
fig.add_trace(go.Scattergeo(
    lon = data_df['lon'],
    lat = data_df['lat'],
    hoverinfo = 'text',
    text = data_df['city'],
    mode = 'markers',  # show text: "markers+text"
    marker = dict(
        size = 2,
        color = 'rgb(255, 0, 0)',
        # bgcolor = 'rgb(255, 255, 255)',
        line = dict(
            width = 1,
            color = 'rgba(255, 0, 0, 0)'

        )
    )))

fig.show()

## Customize the Map

In [6]:
# einkreisen auf Europa: 
fig.update_layout(

    showlegend = False,
    geo = go.layout.Geo(
        scope = 'europe',
        projection_type = 'azimuthal equal area',
        showland = True,
        landcolor = 'rgb(243, 243, 243)',
        countrycolor = 'rgb(204, 204, 204)',# 'rgb(204, 204, 204)',
        
    ),
    height=700,
)

fig.show()

In [7]:
fig.write_html("output/naegeli_map.html")
fig.write_image("output/naegeli_map.svg", format="svg")
fig.write_image("output/naegeli_map.pdf", format="pdf")
